# Whale Instance Segmentation — YOLOv8 on Thermal Grayscale Images

This notebook documents the full pipeline for training and evaluating a YOLOv8 segmentation model on thermal drone images of whales.

**Pipeline overview:**
1. Dataset preparation — RGB → Grayscale conversion
2. Training (with best tuned model)
    - M0: create the new dataset / train, val, test / showing results
    - M1
    - M2

---
## 0. Imports & Global Configuration

In [1]:
# ── Libraries ──────────────────────────────────────────────────────────
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import shutil
import re
import cv2
import matplotlib.patches as mpatches

from collections import Counter
from PIL import Image
from ultralytics import YOLO
from pathlib import Path
from ultralytics.utils.plotting import Annotator, colors

In [2]:
zip_file = "./Flukeprint_detect_behav.v1-2026-05-10-behav.yolov8.zip"
new_name = "05_10_behav_dataset"

# Extract
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(new_name)

print(f"Extracted to {new_name}")

Extracted to 05_10_behav_dataset


In [3]:
# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT  = Path("./05_10_behav_dataset")
DATASET_YAML      = DATASET_ROOT / "data.yaml"

# ── Model weights ─────────────────────────────────────────────────────────────
MODEL_TUNED_PATH    = "./hyper_tuned_best.pt"     # Final tuned model

# ── Model parameters ─────────────────────────────────────────────────────────────
MODEL_PARAMS_PATH = "./best_hyperparameters.yaml"

SPLITS       = ["train", "valid", "test"]

# ── Inference thresholds ─────────────────────────
CONF_THRESHOLD = 0.25  # Confidence threshold
IOU_THRESHOLD  = 0.30  # IoU threshold — intentionally permissive for partial whale detections

---
## 1. Dataset Preparation

### RGB → Grayscale 

The thermal drone images are inherently black-and-white (single-channel), but were saved as 3-channel RGB files by duplicating the same intensity value into R, G and B.

**Why this hurts training:**
- The model wastes capacity learning from redundant colour channels.
- Data augmentation (HSV jitter, colour noise) adds artefacts that don't exist in real thermal imagery, confusing the model.

**Fix:** Convert every image to true 1-channel grayscale before training.

In [5]:
# Convert all RGB images in the dataset to true 1-channel grayscale
# Images are overwritten in-place

total_converted = 0

print("🔄 Starting RGB → Grayscale conversion...")

for split in SPLITS:
    img_dir = Path(DATASET_ROOT) / split / "images"

    if not img_dir.exists():
        print(f"   ⚠️  Skipped '{split}': directory not found at {img_dir}")
        continue

    files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))

    if not files:
        print(f"   ⚠️  Skipped '{split}': no images found.")
        continue

    print(f"   Processing '{split}' ({len(files)} images)...")

    for img_path in files:
        try:
            img = Image.open(img_path)
            if img.mode == "RGB":
                img.convert("L").save(img_path)  # L = 8-bit grayscale
                total_converted += 1
        except Exception as e:
            print(f"   ❌ Error on {img_path.name}: {e}")

print(f"\n✅ Done — {total_converted} images converted to true 1-channel grayscale.")

🔄 Starting RGB → Grayscale conversion...
   Processing 'train' (697 images)...
   Processing 'valid' (149 images)...
   Processing 'test' (149 images)...

✅ Done — 995 images converted to true 1-channel grayscale.


In [6]:
# Verify dataset split sizes and confirm images are truly grayscale

print("📂 Dataset split sizes:")
for split in SPLITS:
    img_dir = Path(DATASET_ROOT) / split / "images"
    images = list(img_dir.rglob("*.jpg")) + list(img_dir.rglob("*.png"))
    print(f"   {split:6s}: {len(images)} images")

# Spot-check one image to confirm channel count
sample_img_path = next((Path(DATASET_ROOT) / "train" / "images").rglob("*.jpg"))
img = Image.open(sample_img_path)
channels = len(img.getbands())
print(f"\n🔍 Sample image: {sample_img_path.name}")
print(f"   Size: {img.width} x {img.height} px")
print(f"   Channels: {channels} ({'✅ Grayscale' if channels == 1 else '❌ Still RGB — rerun conversion'})")

📂 Dataset split sizes:
   train : 697 images
   valid : 149 images
   test  : 149 images

🔍 Sample image: video_013_event7_A007_740_t-00240s_jpg.rf.9a7dd7ac2122dbc08adc0b8ca0b4a8bd.jpg
   Size: 448 x 448 px
   Channels: 1 (✅ Grayscale)


---
## M0 - Binary Detection

### Label transformation

- All classes → 0 (flukeprint)
- Writes to a new dataset folder, never overwrites the original

In [9]:
# ── config ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT  = Path("dataset_M0")
# ─────────────────────────────────────────────────────────────────────────────


def transform_label_file(src: Path, dst: Path) -> tuple[int, int]:
    """
    Read a YOLOv8 polygon label file, remap classes, write to dst.
    Returns (total_annotations, remapped_count).
    """
    lines = src.read_text().splitlines()
    out_lines = []
    total = remapped = 0

    for line in lines:
        line = line.strip()
        if not line:
            out_lines.append(line)
            continue

        parts = line.split()
        cls = int(parts[0])
        total += 1

        parts[0] = "0"
        remapped += 1

        out_lines.append(" ".join(parts))

    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(out_lines))
    return total, remapped


def main():

    grand_total = grand_remapped = 0

    for split in SPLITS:
        img_src = DATASET_ROOT / split / "images"
        lbl_src = DATASET_ROOT / split / "labels"
        img_dst = OUTPUT_ROOT  / split / "images"
        lbl_dst = OUTPUT_ROOT  / split / "labels"

        # ── images: copy as-is ───────────────────────────────────────────────
        if img_src.exists():
            if img_dst.exists():
                shutil.rmtree(img_dst)
            shutil.copytree(img_src, img_dst)
            n_imgs = len(list(img_dst.iterdir()))
            print(f"[{split}] copied {n_imgs} images")
        else:
            print(f"[{split}] no images folder found, skipping")

        # ── labels: remap classes ────────────────────────────────────────────
        if not lbl_src.exists():
            print(f"[{split}] no labels folder found, skipping")
            continue

        lbl_dst.mkdir(parents=True, exist_ok=True)
        split_total = split_remapped = 0

        for lbl_file in sorted(lbl_src.glob("*.txt")):
            t, r = transform_label_file(lbl_file, lbl_dst / lbl_file.name)
            split_total    += t
            split_remapped += r

        grand_total    += split_total
        grand_remapped += split_remapped
        print(f"[{split}] {split_total} annotations — "
              f"{split_remapped} remapped to class 0")

    # ── copy + patch yaml if present ─────────────────────────────────────────
    for yaml_file in DATASET_ROOT.glob("*.yaml"):
        content = yaml_file.read_text()
        content = re.sub(r"nc\s*:\s*\d+", "nc: 1", content)
        content = re.sub(
            r"names\s*:.*?(?=\n\S|\Z)",
            "names:\n  0: flukeprint",
            content,
            flags=re.DOTALL,
        )
        dst_yaml = OUTPUT_ROOT / yaml_file.name
        dst_yaml.parent.mkdir(parents=True, exist_ok=True)
        dst_yaml.write_text(content)
        print(f"[yaml] patched and copied → {dst_yaml}")

    print(f"\n✓ Done. Output: {OUTPUT_ROOT.resolve()}")
    print(f"  Total annotations : {grand_total}")
    print(f"  Remapped → class 0: {grand_remapped}")

In [ ]:
if __name__ == "__main__":
    main()

### M0 Training

In [11]:
# ── Training with best hyperparameters ──────────────────────────────────
# Load the best hyperparameters found by the tuner and train.

# ── config ────────────────────────────────────────────────────────────────────
M0_DATASET  = Path("dataset_M0")
M0_YAML     = M0_DATASET / "data.yaml"
# ─────────────────────────────────────────────────────────────────────────────

with open(MODEL_PARAMS_PATH) as f:
    best_params = yaml.safe_load(f)

model_tuned = YOLO(MODEL_TUNED_PATH)

In [8]:
model_tuned.train(
    data=M0_YAML,
    epochs=150,        
    imgsz=448,
    patience=20,
    name="train_M0",
    **best_params      
)

New https://pypi.org/project/ultralytics/8.4.48 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00784, box=5.14291, cache=False, cfg=None, classes=None, close_mosaic=6, cls=0.55957, compile=False, conf=None, copy_paste=0.00314, copy_paste_mode=flip, cos_lr=False, cutmix=0.00571, data=dataset_M0/data.yaml, degrees=0.00017, deterministic=True, device=None, dfl=1.36906, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.48872, flipud=0.0018, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01493, hsv_s=0.64776, hsv_v=0.27383, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00218, lrf=0.02127, mask_ratio=4, max_det=300, mixup=0.00285, mode=train, model=./hyper_tuned_

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f2ab8eb43a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

### M0 Evaluation

In [12]:
# ── Validation — M0 ───────────────────────────────────────────────

M0_BEST = Path("runs/segment/train_M0/weights/best.pt")  

final_model = YOLO(M0_BEST)

metrics = final_model.val(
    data=M0_YAML,
    split="val",
    conf=CONF_THRESHOLD,
    iou=0.1,
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("M0 VALIDATION RESULTS — Tuned Model (best.pt)")
print("=" * 60)
print(f"mAP50        (Box):  {metrics.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics.box.map:.4f}")
print(f"mAP50        (Mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics.seg.map:.4f}")
print(f"Recall       (Mask): {metrics.seg.r.mean():.4f}")
print(f"Precision    (Mask): {metrics.seg.p.mean():.4f}")
print("=" * 60)


Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 933.0±251.9 MB/s, size: 12.6 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M0/valid/labels.cache... 149 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 24.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.3it/s 1.9s.2s
                   all        149        514      0.858      0.743      0.838      0.693      0.854      0.719      0.826      0.612
Speed: 1.1ms preprocess, 3.0ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /home/floreM/FlukePrint_YOLO/behav_model/runs/segment/val

M0 VALIDATION RESULTS — Tuned Model (best.pt)
mAP50        (Box):  0.8383
mAP50-95     (Box):  0.6

In [7]:
# ── Final Test Evaluation — M0 ───────────────────────────────────────────────

metrics_test = final_model.val(
    data=M0_YAML,
    split="test",
    conf=CONF_THRESHOLD,
    iou=0.1,
    verbose=True,
    plots=True
)

p  = metrics_test.seg.p.mean()
r  = metrics_test.seg.r.mean()
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

print("\n" + "=" * 60)
print("M0 TEST RESULTS — Final Held-Out Evaluation")
print("=" * 60)
print(f"mAP50        (Box):  {metrics_test.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics_test.box.map:.4f}")
print(f"mAP50        (Mask): {metrics_test.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics_test.seg.map:.4f}")
print(f"Recall       (Mask): {r:.4f}")
print(f"Precision    (Mask): {p:.4f}")
print(f"F1           (Mask): {f1:.4f}")
print("=" * 60)
print("✓ M0 baseline locked. Safe to proceed to M1.")

Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 607.7±140.0 MB/s, size: 12.5 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M0/test/labels.cache... 149 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 48.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.4it/s 1.8s.2s
                   all        149        513      0.882      0.743      0.843      0.686      0.855      0.741       0.83      0.603
Speed: 1.4ms preprocess, 1.4ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /home/floreM/FlukePrint_YOLO/behav_model/runs/segment/val2

M0 TEST RESULTS — Final Held-Out Evaluation
mAP50        (Box):  0.8426
mAP50-95     (Box):  0.6859
mAP50        (Mask): 0.8295
mAP50-95     (Mask): 0.6027
Recall       (Mask): 0.7407
Pr

### M0 Visualisation

In [ ]:
# ── config ────────────────────────────────────────────────────────────────────
IMG_DIR      = Path("dataset_M0/test/images")
LBL_DIR      = Path("dataset_M0/test/labels")
N_IMAGES     = 12        # how many to display
COLS         = 2         # side-by-side pairs per row
CLASS_NAMES  = {0: "flukeprint"}
GT_COLOR     = (0, 255, 0)    # green  — ground truth
PRED_COLOR   = (255, 80, 80)  # red    — prediction
# ─────────────────────────────────────────────────────────────────────────────

img_paths = sorted(IMG_DIR.glob("*.jpg")) + sorted(IMG_DIR.glob("*.png"))
img_paths = img_paths[:N_IMAGES]


def draw_gt(img, lbl_path, color=GT_COLOR):
    """Draw ground-truth polygons from a YOLO label file."""
    h, w = img.shape[:2]
    if not lbl_path.exists():
        return img
    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        coords = list(map(float, parts[1:]))
        pts = np.array([(coords[i] * w, coords[i+1] * h)
                        for i in range(0, len(coords), 2)], dtype=np.int32)
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=2)
    return img


def draw_pred(img, result, color=PRED_COLOR):
    """Draw predicted masks/polygons from a YOLO result."""
    if result.masks is None:
        return img
    h, w = img.shape[:2]
    for mask_xy in result.masks.xy:
        pts = mask_xy.astype(np.int32)
        cv2.polylines(img, [pts], isClosed=True, color=color, thickness=2)
    return img


# ── plot ──────────────────────────────────────────────────────────────────────
n      = len(img_paths)
rows   = n  # one row per image
fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
if n == 1:
    axes = [axes]

for ax_row, img_path in zip(axes, img_paths):
    lbl_path = LBL_DIR / (img_path.stem + ".txt")

    # ground truth
    img_gt = cv2.imread(str(img_path))
    img_gt = cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB)
    img_gt = draw_gt(img_gt.copy(), lbl_path)

    # prediction
    img_pr = cv2.imread(str(img_path))
    img_pr = cv2.cvtColor(img_pr, cv2.COLOR_BGR2RGB)
    result  = final_model.predict(img_path, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)[0]
    img_pr  = draw_pred(img_pr.copy(), result)

    ax_row[0].imshow(img_gt)
    ax_row[0].set_title(f"GT — {img_path.name}", fontsize=8)
    ax_row[0].axis("off")

    ax_row[1].imshow(img_pr)
    ax_row[1].set_title(f"Pred (conf≥{CONF_THRESHOLD}) — {img_path.name}", fontsize=8)
    ax_row[1].axis("off")

# legend
gt_patch   = mpatches.Patch(color=(0,1,0),     label="Ground truth")
pred_patch = mpatches.Patch(color=(1,.3,.3),   label="Predicted")
fig.legend(handles=[gt_patch, pred_patch], loc="upper right", fontsize=10)
fig.suptitle("M0 — Ground Truth vs Prediction", fontsize=13, fontweight="bold", y=1.001)
plt.tight_layout()
plt.savefig("m0_gt_vs_pred.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → m0_gt_vs_pred.png")

<Figure size 1200x4800 with 24 Axes>

Saved → m0_gt_vs_pred.png


---
## M1 - Behavioral Distinction

### Label transformation

Original 9 classes → 4 behavioral classes (background is implicit, never labeled)

    0: feeding_fresh          → 0: Feeding
    1: feeding_old            → 0: Feeding
    2: surfacing_fresh        → 1: Surfacing
    3: surfacing_fresh_travel → 1: Surfacing
    4: surfacing_old          → 1: Surfacing
    5: surfacing_old_travel   → 1: Surfacing
    6: traveling_fresh        → 2: Traveling
    7: traveling_old          → 2: Traveling
    8: unknown                → 3: Unknown

Writes to a new dataset folder, never overwrites the original

In [7]:
# ── config ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT  = Path("dataset_M1")

CLASS_MAP = {
    0: 0,   # feeding_fresh          → Feeding
    1: 0,   # feeding_old            → Feeding
    2: 1,   # surfacing_fresh        → Surfacing
    3: 1,   # surfacing_fresh_travel → Surfacing
    4: 1,   # surfacing_old          → Surfacing
    5: 1,   # surfacing_old_travel   → Surfacing
    6: 2,   # traveling_fresh        → Traveling
    7: 2,   # traveling_old          → Traveling
    8: 3,   # unknown                → Unknown
}

CLASS_NAMES = {
    0: "feeding",
    1: "surfacing",
    2: "traveling",
    3: "unknown",
}
# ─────────────────────────────────────────────────────────────────────────────


def transform_label_file(src: Path, dst: Path) -> Counter:
    """
    Remap class IDs in a YOLOv8 polygon label file.
    Returns a Counter of {new_class: count}.
    """
    lines = src.read_text().splitlines()
    out_lines = []
    counts = Counter()

    for line in lines:
        line = line.strip()
        if not line:
            out_lines.append(line)
            continue

        parts = line.split()
        original_cls = int(parts[0])

        if original_cls not in CLASS_MAP:
            raise ValueError(
                f"Unexpected class {original_cls} in {src}. "
                f"Expected one of {list(CLASS_MAP.keys())}"
            )

        new_cls = CLASS_MAP[original_cls]
        parts[0] = str(new_cls)
        counts[new_cls] += 1
        out_lines.append(" ".join(parts))

    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(out_lines))
    return counts


def main():
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(f"DATASET_ROOT not found: {DATASET_ROOT.resolve()}")

    if OUTPUT_ROOT.exists():
        print(f"[warn] Output folder already exists: {OUTPUT_ROOT.resolve()}")
        print("       Delete it manually if you want a clean run.")

    grand_counts = Counter()

    for split in SPLITS:
        img_src = DATASET_ROOT / split / "images"
        lbl_src = DATASET_ROOT / split / "labels"
        img_dst = OUTPUT_ROOT  / split / "images"
        lbl_dst = OUTPUT_ROOT  / split / "labels"

        # ── images: copy as-is ───────────────────────────────────────────────
        if img_src.exists():
            if img_dst.exists():
                shutil.rmtree(img_dst)
            shutil.copytree(img_src, img_dst)
            n_imgs = len(list(img_dst.iterdir()))
            print(f"[{split}] copied {n_imgs} images")
        else:
            print(f"[{split}] no images folder found, skipping")

        # ── labels: remap classes ────────────────────────────────────────────
        if not lbl_src.exists():
            print(f"[{split}] no labels folder found, skipping")
            continue

        lbl_dst.mkdir(parents=True, exist_ok=True)
        split_counts = Counter()

        for lbl_file in sorted(lbl_src.glob("*.txt")):
            counts = transform_label_file(lbl_file, lbl_dst / lbl_file.name)
            split_counts += counts

        grand_counts += split_counts

        print(f"[{split}] annotation breakdown:")
        for cls_id, name in CLASS_NAMES.items():
            print(f"         class {cls_id} ({name:12s}): {split_counts[cls_id]}")

    # ── write yaml ───────────────────────────────────────────────────────────
    for yaml_file in DATASET_ROOT.glob("*.yaml"):
        content = yaml_file.read_text()
        content = re.sub(r"nc\s*:\s*\d+", "nc: 4", content)
        content = re.sub(
            r"names\s*:.*?(?=\n\S|\Z)",
            (
                "names:\n"
                "  0: feeding\n"
                "  1: surfacing\n"
                "  2: traveling\n"
                "  3: unknown"
            ),
            content,
            flags=re.DOTALL,
        )
        dst_yaml = OUTPUT_ROOT / yaml_file.name
        dst_yaml.parent.mkdir(parents=True, exist_ok=True)
        dst_yaml.write_text(content)
        print(f"\n[yaml] patched and copied → {dst_yaml}")

    print(f"\n✓ Done. Output: {OUTPUT_ROOT.resolve()}")
    print(f"\nGlobal annotation breakdown:")
    total = 0
    for cls_id, name in CLASS_NAMES.items():
        n = grand_counts[cls_id]
        total += n
        print(f"  class {cls_id} ({name:12s}): {n:>6} annotations")
    print(f"  {'TOTAL':>18}: {total:>6} annotations")

In [ ]:
if __name__ == "__main__":
    main()

### M1 Training

In [8]:
# ── Training with best hyperparameters ──────────────────────────────────
# Load the best hyperparameters found by the tuner and train.

# ── config ────────────────────────────────────────────────────────────────────
M1_DATASET      = Path("dataset_M1")
M1_YAML         = M1_DATASET / "data.yaml"
M0_BEST         = Path("runs/segment/train_M0/weights/best.pt")  # ← change this
# ─────────────────────────────────────────────────────────────────────────────

model_m1 = YOLO(M0_BEST)  # ← and this

with open(MODEL_PARAMS_PATH) as f:
    best_params = yaml.safe_load(f)

# Override cls loss — tuned for binary, needs boosting for 4-class imbalance
best_params["cls"] = 1   # up from 0.55, helps rare classes (traveling, unknown) // bis 1.0 less agressive

In [9]:
model_m1.train(
    data=M1_YAML,
    epochs=200,  # 150 / bis 200 epochs
    imgsz=448,
    patience=40,  # 20 / bis 40 patience
    name="train_M1_bis",
    freeze=None, # bis new, unfreeze all layers for full fine-tuning
    **best_params       # cls=1.5 now injected here / bis 1
)

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00784, box=5.14291, cache=False, cfg=None, classes=None, close_mosaic=6, cls=1, compile=False, conf=None, copy_paste=0.00314, copy_paste_mode=flip, cos_lr=False, cutmix=0.00571, data=dataset_M1/data.yaml, degrees=0.00017, deterministic=True, device=None, dfl=1.36906, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.48872, flipud=0.0018, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01493, hsv_s=0.64776, hsv_v=0.27383, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00218, lrf=0.02127, mask_ratio=4, max_det=300, mixup=0.00285, mode=train, model=runs/segment/train_M

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4eb2128fd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0

### M1 Evaluation

In [10]:
# ── Validation — M1 ───────────────────────────────────────────────

M1_BEST = Path("runs/segment/train_M1_bis/weights/best.pt")  

final_model = YOLO(M1_BEST)

metrics = final_model.val(
    data=M1_YAML,
    split="val",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("M1 BIS VALIDATION RESULTS — Tuned Model (best.pt)")
print("=" * 60)
print(f"mAP50        (Box):  {metrics.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics.box.map:.4f}")
print(f"mAP50        (Mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics.seg.map:.4f}")
print(f"Recall       (Mask): {metrics.seg.r.mean():.4f}")
print(f"Precision    (Mask): {metrics.seg.p.mean():.4f}")
print("=" * 60)


Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)


YOLOv8s-seg summary (fused): 86 layers, 11,781,148 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 799.3±201.9 MB/s, size: 10.0 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M1/valid/labels.cache... 149 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 41.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.4it/s 1.9s.2s
                   all        149        514      0.596      0.653      0.622      0.469      0.519      0.579      0.531       0.36
               feeding         69        150      0.715      0.819      0.845      0.722       0.72      0.807      0.847      0.673
             surfacing        104        278       0.67      0.745      0.719      0.562      0.685      0.755      0.728      0.485
             traveling         33         44      0.602        0.5      0.558      0.324

In [11]:
# ── Final Test Evaluation — M1 ───────────────────────────────────────────────

metrics_test = final_model.val(
    data=M1_YAML,
    split="test",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

p  = metrics_test.seg.p.mean()
r  = metrics_test.seg.r.mean()
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

print("\n" + "=" * 60)
print("M1 BIS TEST RESULTS — Final Held-Out Evaluation")
print("=" * 60)
print(f"mAP50        (Box):  {metrics_test.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics_test.box.map:.4f}")
print(f"mAP50        (Mask): {metrics_test.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics_test.seg.map:.4f}")
print(f"Recall       (Mask): {r:.4f}")
print(f"Precision    (Mask): {p:.4f}")
print(f"F1           (Mask): {f1:.4f}")
print("=" * 60)
# print("✓ M1 baseline locked. Safe to proceed to M2.")

Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1043.8±333.2 MB/s, size: 12.4 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M1/test/labels.cache... 149 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 44.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.8it/s 1.7s.1s
                   all        149        513      0.654      0.563      0.614      0.472      0.548      0.421      0.453      0.312
               feeding         65        151      0.823      0.682      0.819      0.691       0.84      0.642      0.813      0.649
             surfacing        105        295      0.801      0.732      0.787      0.597      0.818      0.671      0.747      0.499
             traveling         34         47      0.843      0.638      0.741 

### M1 Visualisation

In [12]:
# ── config ────────────────────────────────────────────────────────────────────
IMG_DIR   = Path("dataset_M1/test/images")
LBL_DIR   = Path("dataset_M1/test/labels")
N_IMAGES  = 12
CLASS_NAMES = {0: "feeding", 1: "surfacing", 2: "traveling", 3: "unknown"}

CLASS_COLORS_BGR = {
    0: (0,   200,  50),    # feeding   → vivid green
    1: (255,  80,   0),    # surfacing → orange
    2: (0,   100, 255),    # traveling → blue
    3: (180,   0, 255),    # unknown   → purple
}

CLASS_COLORS_RGB = {
    k: tuple(c / 255 for c in reversed(v))
    for k, v in CLASS_COLORS_BGR.items()
}
# ─────────────────────────────────────────────────────────────────────────────

img_paths = sorted(IMG_DIR.glob("*.jpg")) + sorted(IMG_DIR.glob("*.png"))
img_paths = img_paths[:N_IMAGES]


def draw_gt(img_bgr: np.ndarray, lbl_path: Path) -> np.ndarray:
    """Draw GT polygons using Ultralytics Annotator — one color per class."""
    annotator = Annotator(img_bgr.copy(), line_width=2, font_size=10)
    h, w = img_bgr.shape[:2]

    if not lbl_path.exists():
        return annotator.result()

    for line in lbl_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls_id = int(parts[0])
        coords = list(map(float, parts[1:]))
        pts = np.array(
            [(coords[i] * w, coords[i + 1] * h) for i in range(0, len(coords), 2)],
            dtype=np.int32,
        )
        color = CLASS_COLORS_BGR[cls_id]
        label = CLASS_NAMES.get(cls_id, str(cls_id))
        # draw filled polygon + outline
        cv2.polylines(annotator.im, [pts], isClosed=True, color=color, thickness=2)
        cv2.fillPoly(annotator.im, [pts], color=(*color[:3], 60))  # semi-transparent
        # label at centroid
        cx, cy = pts[:, 0].mean().astype(int), pts[:, 1].mean().astype(int)
        annotator.text((cx, cy), label, txt_color=(255, 255, 255))

    return annotator.result()


def draw_pred(img_bgr: np.ndarray, result) -> np.ndarray:
    """Draw predictions with custom per-class colors."""
    out = img_bgr.copy()
    if result.masks is None:
        return out
    for mask_xy, box in zip(result.masks.xy, result.boxes):
        cls_id = int(box.cls)
        conf   = float(box.conf)
        color  = CLASS_COLORS_BGR[cls_id]
        label  = f"{CLASS_NAMES.get(cls_id, cls_id)} {conf:.2f}"
        pts    = mask_xy.astype(np.int32)

        # filled mask + outline
        overlay = out.copy()
        cv2.fillPoly(overlay, [pts], color=color)
        cv2.addWeighted(overlay, 0.3, out, 0.7, 0, out)  # semi-transparent fill
        cv2.polylines(out, [pts], isClosed=True, color=color, thickness=2)

        # label at centroid
        cx, cy = pts[:, 0].mean().astype(int), pts[:, 1].mean().astype(int)
        cv2.putText(out, label, (cx, cy), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(N_IMAGES, 2, figsize=(14, 5 * N_IMAGES))
if N_IMAGES == 1:
    axes = [axes]

for ax_row, img_path in zip(axes, img_paths):
    lbl_path = LBL_DIR / (img_path.stem + ".txt")

    img_bgr = cv2.imread(str(img_path))
    result  = final_model.predict(img_path, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)[0]

    img_gt   = draw_gt(img_bgr, lbl_path)
    img_pred = draw_pred(img_bgr, result)

    # GT counts for title
    gt_lines = [l for l in lbl_path.read_text().splitlines() if l.strip()] if lbl_path.exists() else []
    n_gt   = len(gt_lines)
    n_pred = len(result.boxes) if result.boxes is not None else 0

    ax_row[0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
    ax_row[0].set_title(f"GT ({n_gt} annotations) — {img_path.name}", fontsize=8, pad=4)
    ax_row[0].axis("off")

    ax_row[1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
    ax_row[1].set_title(f"Pred ({n_pred} detections, conf≥{CONF_THRESHOLD}) — {img_path.name}", fontsize=8, pad=4)
    ax_row[1].axis("off")

# ── legend — one patch per class, matching Ultralytics colors ────────────────
patches = [
    mpatches.Patch(color=CLASS_COLORS_RGB[i], label=name)
    for i, name in CLASS_NAMES.items()
]
fig.legend(handles=patches, loc="upper right", fontsize=10, title="Classes", framealpha=0.9)
fig.suptitle("M1 bis — Ground Truth vs Prediction (Behavioral Classes)", fontsize=13, fontweight="bold", y=1.001)

plt.tight_layout()
out_path = "m1_bis_gt_vs_pred.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

<Figure size 1400x6000 with 24 Axes>

Saved → m1_bis_gt_vs_pred.png


### test M1 yolov8s

In [14]:
with open(MODEL_PARAMS_PATH) as f:
    best_params = yaml.safe_load(f)

# Override cls loss — tuned for binary, needs boosting for 4-class imbalance
best_params["cls"] = 1.5   # up from 0.55, helps rare classes (traveling, unknown) // bis 1.0 less agressive

print(best_params["cls"])

1.5


In [ ]:
model_yolov8s = YOLO("yolov8s-seg.pt")

model_yolov8s.train(
    data=M1_YAML,
    epochs=200,  
    imgsz=448,
    patience=40,  
    name="train_M1_yolov8s",
    **best_params       # cls=1.5 now injected here 
)

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00784, box=5.14291, cache=False, cfg=None, classes=None, close_mosaic=6, cls=1.5, compile=False, conf=None, copy_paste=0.00314, copy_paste_mode=flip, cos_lr=False, cutmix=0.00571, data=dataset_M1/data.yaml, degrees=0.00017, deterministic=True, device=None, dfl=1.36906, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.48872, flipud=0.0018, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01493, hsv_s=0.64776, hsv_v=0.27383, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00218, lrf=0.02127, mask_ratio=4, max_det=300, mixup=0.00285, mode=train, model=yolov8s-seg.pt, mo

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4b763732b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0

In [16]:
# ── Validation — M1 ───────────────────────────────────────────────

M1_yolov8s_BEST = Path("runs/segment/train_M1_yolov8s/weights/best.pt")  

final_model = YOLO(M1_yolov8s_BEST)

metrics = final_model.val(
    data=M1_YAML,
    split="val",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("M1 YOLOv8s VALIDATION RESULTS — Tuned Model (best.pt)")
print("=" * 60)
print(f"mAP50        (Box):  {metrics.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics.box.map:.4f}")
print(f"mAP50        (Mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics.seg.map:.4f}")
print(f"Recall       (Mask): {metrics.seg.r.mean():.4f}")
print(f"Precision    (Mask): {metrics.seg.p.mean():.4f}")
print("=" * 60)


Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,781,148 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 679.5±164.9 MB/s, size: 10.0 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M1/valid/labels.cache... 149 images, 20 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 34.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 4.7it/s 2.1s.3s
                   all        149        514      0.619      0.613      0.632      0.479      0.475      0.501      0.492      0.355
               feeding         69        150       0.74      0.833      0.831      0.686      0.746       0.84      0.834      0.636
             surfacing        104        278      0.689      0.741      0.726      0.576      0.709      0.763      0.741  

In [17]:
# ── Final Test Evaluation — M1 ───────────────────────────────────────────────

metrics_test = final_model.val(
    data=M1_YAML,
    split="test",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

p  = metrics_test.seg.p.mean()
r  = metrics_test.seg.r.mean()
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

print("\n" + "=" * 60)
print("M1 YOLOv8s TEST RESULTS — Final Held-Out Evaluation")
print("=" * 60)
print(f"mAP50        (Box):  {metrics_test.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics_test.box.map:.4f}")
print(f"mAP50        (Mask): {metrics_test.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics_test.seg.map:.4f}")
print(f"Recall       (Mask): {r:.4f}")
print(f"Precision    (Mask): {p:.4f}")
print(f"F1           (Mask): {f1:.4f}")
print("=" * 60)
# print("✓ M1 baseline locked. Safe to proceed to M2.")

Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 644.7±239.6 MB/s, size: 11.3 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/behav_model/dataset_M1/test/labels.cache... 149 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 22.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 4.7it/s 2.1s.1s
                   all        149        513      0.598      0.602      0.617      0.479      0.441       0.47      0.455       0.33
               feeding         65        151      0.765      0.755      0.818      0.674      0.772      0.762      0.818      0.639
             surfacing        105        295      0.755      0.793       0.77      0.577      0.726      0.763      0.744      0.496
             traveling         34         47      0.756       0.66      0.709  

### test M1 yolo26s

In [18]:
model_yolo26s = YOLO("yolo26s-seg.pt")

model_yolo26s.train(
    data=M1_YAML,
    epochs=200,  
    imgsz=448,
    patience=40,  
    name="train_M1_yolo26s",
    **best_params       # cls=1.5 now injected here 
)

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00784, box=5.14291, cache=False, cfg=None, classes=None, close_mosaic=6, cls=1.5, compile=False, conf=None, copy_paste=0.00314, copy_paste_mode=flip, cos_lr=False, cutmix=0.00571, data=dataset_M1/data.yaml, degrees=0.00017, deterministic=True, device=None, dfl=1.36906, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.48872, flipud=0.0018, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01493, hsv_s=0.64776, hsv_v=0.27383, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00218, lrf=0.02127, mask_ratio=4, max_det=300, mixup=0.00285, mode=train, model=yolo26s-seg.pt, mo

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4b5e1f20e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0